# Phase 3a: Raster exports from Google Earth Engine

This notebook asks Google Earth Engine (GEE) to make the satellite rasters for the fields and save them to Google Drive (folder `thesis_phase3`). GEE does the work on Google's computers. The notebook only sends the jobs (a few minutes), then GEE processes them in the background (a few hours).

**What gets exported**
| Part | What | Area | Pixel |
|---|---|---|---|
| A | Sentinel-1 filter test: one August 2025 date, 6 versions (no filter + 5 filters) | 3 context boxes | 10 m |
| A | Sentinel-1 filter test: field means for every filter and month of 2025 | 2025 fields | table |
| A | Sentinel-2: 7 indices + number of clear observations, monthly | fields + 30 m strip | 10 m |
| A | Sentinel-2 NDVI, monthly, for maps | 3 context boxes | 10 m |
| A | Sentinel-2 clear observations per field and month | all fields | table |
| A | MODIS LAI, FPAR, GPP, PsnNet, monthly | export area | 500 m |
| A | ERA5-Land temperature, VPD, rain, evaporation, soil moisture, monthly | export area | 10 km |
| A | CHIRPS rain, monthly and season | export area | 5 km |
| A | Elevation, slope, aspect, TWI; SoilGrids; WorldCover | export area | 30 m, 250 m, 10 m |
| B | **Sentinel-1 main** (VV, VH, VH/VV, RVI, NDPI, number of images), monthly, with the chosen filter | fields + 30 m strip, and 3 boxes | 10 m |

Part B runs only after the filter is chosen in P3b.

**Before you run**
1. `pip install earthengine-api` in the .venv.
2. Put `s1_speckle.py` in the `src/` folder (create it if needed). It is the published speckle filter code from Mullissa et al. (2021), MIT licence, unchanged.
3. Step 0 logs in by itself the first time (a browser window opens). Use the Google account that owns the project `ee-josaiahsyiem`.

All 10 m exports use one fixed grid (EPSG:32643), so every file lines up pixel by pixel.

## Step 0: Setup

In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import box
import ee

here = Path.cwd()
ROOT = here if (here / "data" / "raw").exists() else here.parent
DATA2 = ROOT / "data" / "processed_v2"
OUT = ROOT / "results" / "phase3"; OUT.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(ROOT / "src"))
import s1_speckle as sf                       # published speckle filters (Mullissa et al. 2021)

PROJECT = "ee-josaiahsyiem"
DRIVE_FOLDER = "thesis_phase3"                # all exports go to this Google Drive folder
CRS = "EPSG:32643"
YEARS = [2022, 2023, 2024, 2025]
MONTHS = [6, 7, 8, 9, 10, 11]                 # June to November
BUFFER_M = 30                                 # strip kept around each field
KERNEL = 5                                    # window for Lee sigma and Gamma MAP (5 x 5 pixels = 50 m)

try:
    ee.Initialize(project=PROJECT)
except Exception:
    ee.Authenticate()                         # opens the browser the first time; remembered after
    ee.Initialize(project=PROJECT)
print("Earth Engine ready, project:", PROJECT)

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


Earth Engine ready, project: ee-josaiahsyiem


## Step 1: Fields to GEE and the export area

The 1,205 fields from `yield_analysis_ready_v2.gpkg` are sent to GEE (simplified by 1 m to keep the request small). The export area is the box around all fields plus 1 km. For each year, a mask keeps only that year's fields plus a 30 m strip, which texture and filters need at the field edges.

**Found:** 1,205 fields sent (55, 60, 323 and 767 per year); export area 86.0 × 81.0 km. P3b and later phases use 1,202 fields, because the 3 fields with impossible yields were removed in Phase 1B; that does not change the rasters.

In [2]:
fields = gpd.read_file(DATA2 / "yield_analysis_ready_v2.gpkg")[["uid", "year", "geometry"]]
fields["geometry"] = fields.geometry.simplify(1)                 # remove extra vertices (1 m), keeps the shape
cent = fields.copy(); cent["geometry"] = fields.geometry.representative_point()

def to_ee(gdf):
    return ee.FeatureCollection(json.loads(gdf.to_crs(4326).to_json(drop_id=True)))

fc_fields = to_ee(fields)
fc_points = to_ee(cent)
print("fields sent:", fc_fields.size().getInfo(), "| per year:", fields["year"].value_counts().sort_index().to_dict())

# one fixed 10 m grid for every 10 m export, so all files line up pixel by pixel
xmin, ymin, xmax, ymax = fields.total_bounds
xmin, ymin = np.floor((xmin - 1000) / 10) * 10, np.floor((ymin - 1000) / 10) * 10
xmax, ymax = np.ceil((xmax + 1000) / 10) * 10, np.ceil((ymax + 1000) / 10) * 10
GRID10 = [10, 0, xmin, 0, -10, ymax]
REGION = ee.Geometry.Rectangle([xmin, ymin, xmax, ymax], proj=CRS, geodesic=False)
print(f"export area: {(xmax - xmin) / 1000:.1f} x {(ymax - ymin) / 1000:.1f} km")

def field_mask(year):
    """1 inside the fields of that year plus a 30 m strip, masked elsewhere."""
    fc = fc_fields.filter(ee.Filter.eq("year", year)).map(lambda f: f.buffer(BUFFER_M))
    return ee.Image(0).byte().paint(fc, 1).selfMask()

fields sent: 1205 | per year: {2022: 55, 2023: 60, 2024: 323, 2025: 767}
export area: 86.0 x 81.0 km


## Step 2: Context boxes

Three boxes around the Phase 2 hot and cold spots, each the spot's fields plus 2 km on every side. These get full (unmasked) 10 m rasters for the filter test and the detailed maps. Saved as `results/phase3/P3_context_boxes.gpkg` for QGIS.

**Found:** killari 19 × 13 km (15 spot fields, 17 fields in 2024), westlatur 10 × 17 km (13 spot fields; 21, 23, 81 and 106 fields in 2022 to 2025), shirur 9 × 6 km (16 spot fields, 20 fields in 2025).

In [3]:
sp = gpd.read_file(DATA2 / "yield_spatial_v2.gpkg")
pts = sp.geometry.representative_point()
spots = {"killari": (sp["year"] == 2024) & (sp["lisa_class"] == "High-High"),
         "westlatur": (sp["year"] == 2024) & (sp["lisa_class"] == "Low-Low") & (pts.x < 660000),
         "shirur": (sp["year"] == 2025) & (sp["lisa_class"] == "Low-Low")}
rows = []
for name, m in spots.items():
    x0, y0, x1, y1 = pts[m].total_bounds                               # the spot's fields, plus 2 km on every side
    b = box(*(np.round(np.array([x0 - 2000, y0 - 2000, x1 + 2000, y1 + 2000]) / 10) * 10))
    rows.append({"box": name, "spot_fields": int(m.sum()), "size_km": f"{b.bounds[2]/1000 - b.bounds[0]/1000:.0f} x {b.bounds[3]/1000 - b.bounds[1]/1000:.0f}", "geometry": b,
                 "fields_inside": str({y: int((fields[fields["year"] == y].within(b)).sum()) for y in YEARS})})
boxes = gpd.GeoDataFrame(rows, crs=CRS)
boxes.to_file(OUT / "P3_context_boxes.gpkg", driver="GPKG")
print(boxes.drop(columns="geometry").to_string(index=False))
BOX = {r["box"]: ee.Geometry.Rectangle(list(r["geometry"].bounds), proj=CRS, geodesic=False) for _, r in boxes.iterrows()}

      box  spot_fields size_km                             fields_inside
  killari           15 19 x 13     {2022: 0, 2023: 0, 2024: 17, 2025: 0}
westlatur           13 10 x 17 {2022: 21, 2023: 23, 2024: 81, 2025: 106}
   shirur           16   9 x 6     {2022: 0, 2023: 1, 2024: 0, 2025: 20}


## Step 3: Sentinel-1 orbit check

Sentinel-1 looks at Latur from different relative orbits, each at a different angle, so their values are not directly comparable. This step lists every image from June to November of each year (pass, orbit, satellite, date) and how many fields each orbit covers. We keep **one descending orbit** (as in v1): the one that covers the most fields.

If the warning appears, the orbit does not cover every field; stop and decide before going on.

**Found:** all June to November images in the export area come from **one descending orbit, 63**, and one satellite (Sentinel-1A; Sentinel-1B stopped in December 2021). It covers every field in every year (55, 60, 323 and 767), with 14 or 15 dates per season. No orbit choice was needed.

In [4]:
def s1_raw(start, end, region=None):
    return (ee.ImageCollection("COPERNICUS/S1_GRD_FLOAT")          # linear power, with the incidence angle band
            .filterBounds(region or REGION).filterDate(start, end)
            .filter(ee.Filter.eq("instrumentMode", "IW"))
            .filter(ee.Filter.eq("resolution_meters", 10))
            .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
            .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH")))

rows, cov = [], []
for yr in YEARS:
    c = s1_raw(f"{yr}-06-01", f"{yr}-12-01")
    lst = c.reduceColumns(ee.Reducer.toList(4), ["orbitProperties_pass", "relativeOrbitNumber_start",
                                                 "platform_number", "system:time_start"]).get("list").getInfo()
    for p, o, s, t in lst:
        rows.append({"year": yr, "pass": p, "orbit": int(o), "satellite": "S1" + s, "date": pd.to_datetime(t, unit="ms").date()})
    pts_y = fc_points.filter(ee.Filter.eq("year", yr))
    for p, o in {(r[0], int(r[1])) for r in lst}:
        foot = c.filter(ee.Filter.eq("orbitProperties_pass", p)).filter(ee.Filter.eq("relativeOrbitNumber_start", o)).geometry()
        cov.append({"year": yr, "pass": p, "orbit": o, "fields": len(fields[fields["year"] == yr]),
                    "fields_covered": pts_y.filterBounds(foot).size().getInfo()})
imgs = pd.DataFrame(rows).drop_duplicates(subset=["year", "pass", "orbit", "date", "satellite"])
imgs.to_csv(OUT / "P3_s1_images.csv", index=False)
cov = pd.DataFrame(cov)
dates = imgs.groupby(["pass", "orbit", "year"])["date"].nunique().rename("dates")
sats = imgs.groupby(["pass", "orbit", "year"])["satellite"].agg(lambda s: ", ".join(sorted(set(s))))
table = cov.set_index(["pass", "orbit", "year"]).join(dates).join(sats).reset_index().sort_values(["pass", "orbit", "year"])
table.to_csv(OUT / "P3_s1_orbits.csv", index=False)
print(table.to_string(index=False))

PASS = "DESCENDING"                                             # same as the v1 extraction
d = table[table["pass"] == PASS].groupby("orbit").agg(covered=("fields_covered", "sum"), dates=("dates", "sum"))
ORBIT = int(d.sort_values(["covered", "dates"], ascending=False).index[0])
share = table[(table["pass"] == PASS) & (table["orbit"] == ORBIT)].set_index("year")
print(f"\nchosen: {PASS} orbit {ORBIT} | fields covered per year:",
      {y: f"{int(r.fields_covered)} of {int(r.fields)}" for y, r in share.iterrows()})
if (share["fields_covered"] < share["fields"]).any():
    print("WARNING: this orbit does not cover every field. Stop here and paste this output.")

      pass  orbit  year  fields  fields_covered  dates satellite
DESCENDING     63  2022      55              55     15       S1A
DESCENDING     63  2023      60              60     14       S1A
DESCENDING     63  2024     323             323     14       S1A
DESCENDING     63  2025     767             767     14       S1A

chosen: DESCENDING orbit 63 | fields covered per year: {2022: '55 of 55', 2023: '60 of 60', 2024: '323 of 323', 2025: '767 of 767'}


## Step 4: Sentinel-1 functions

- Source: `COPERNICUS/S1_GRD_FLOAT` (linear power). Monthly values are the **mean in linear power, then converted to dB**.
- Bands: VV and VH (dB), VH/VV (dB difference), RVI = 4 VH / (VV + VH), NDPI = (VV − VH) / (VV + VH), and `n_images` (how many images went into that month).
- Filters: `none`, `focal30` (v1: mean within 30 m), `refinedlee`, `leesigma`, `gammamap` and `quegan` (multi-temporal, using 10 earlier images with Gamma MAP). Lee sigma and Gamma MAP use a 5 × 5 window (50 m), because the fields are small (about 1 ha).

In [5]:
def s1_season(year, variant, region=None):
    """June to November images of the chosen orbit, speckle-filtered, linear power (VV, VH)."""
    c = (s1_raw(f"{year}-06-01", f"{year}-12-01", region)
         .filter(ee.Filter.eq("orbitProperties_pass", PASS))
         .filter(ee.Filter.eq("relativeOrbitNumber_start", ORBIT))
         .select(["VV", "VH", "angle"])
         .map(lambda im: im.updateMask(im.select("VV").gt(1e-4))))     # drop empty edge pixels
    if variant == "none":
        out = c
    elif variant == "focal30":                                         # the v1 filter: mean within 30 m
        out = c.map(lambda im: im.addBands(im.select(["VV", "VH"]).focalMean(30, "circle", "meters"), None, True))
    elif variant == "refinedlee":
        out = sf.MonoTemporal_Filter(c, KERNEL, "REFINED LEE")
    elif variant == "leesigma":
        out = sf.MonoTemporal_Filter(c, KERNEL, "LEE SIGMA")
    elif variant == "gammamap":
        out = sf.MonoTemporal_Filter(c, KERNEL, "GAMMA MAP")
    elif variant == "quegan":                                          # multi-temporal, 10 earlier images
        out = sf.MultiTemporal_Filter(c, KERNEL, "GAMMA MAP", 10)
    return ee.ImageCollection(out).select(["VV", "VH"])

def month_range(year, m):
    start = ee.Date.fromYMD(year, m, 1)
    return start, start.advance(1, "month")

def s1_month(coll, year, m):
    """Monthly mean in linear power, then dB. Bands: VV, VH, VH_VV (dB), RVI, NDPI, n_images."""
    c = coll.filterDate(*month_range(year, m))
    lin = c.mean(); vv, vh = lin.select("VV"), lin.select("VH")
    db = lin.log10().multiply(10)
    return (db.select(["VV", "VH"])
            .addBands(db.select("VH").subtract(db.select("VV")).rename("VH_VV"))
            .addBands(vh.multiply(4).divide(vv.add(vh)).rename("RVI"))
            .addBands(vv.subtract(vh).divide(vv.add(vh)).rename("NDPI"))
            .addBands(c.select("VV").count().rename("n_images"))
            .toFloat())

TASKS = []
def export_image(img, name, region, crs_transform=None, scale=None):
    kw = dict(image=img, description=name, folder=DRIVE_FOLDER, fileNamePrefix=name, region=region,
              crs=CRS, maxPixels=1e11, fileFormat="GeoTIFF")
    kw.update({"crsTransform": crs_transform} if crs_transform else {"scale": scale})
    t = ee.batch.Export.image.toDrive(**kw); t.start(); TASKS.append((name, t))

def export_table(fc, name, columns):
    t = ee.batch.Export.table.toDrive(collection=fc, description=name, folder=DRIVE_FOLDER,
                                      fileNamePrefix=name, fileFormat="CSV", selectors=columns)
    t.start(); TASKS.append((name, t))

def tag(props):
    return lambda ft: ft.set(props).setGeometry(None)
print("functions ready")

functions ready


## Step 5: Part A, the speckle filter test

- **(a)** One August 2025 date with all 6 versions, in linear power, for the 3 boxes. P3b uses these for the ENL (in uniform areas), edge sharpness and the visual check.
- **(b)** The mean VV and VH of every 2025 field for every filter and month, one table per filter. P3b uses this for the link with yield. (A first version as one big table failed with "User memory limit exceeded", so it is split by filter and uses `tileScale = 8`.)

**Found:** test date 1 August 2025 (`20250801`). 3 box rasters and 6 field-mean tables were exported; all completed.

In [ ]:
VARIANTS = ["none", "focal30", "refinedlee", "leesigma", "gammamap", "quegan"]
TEST_YEAR = 2025

# (a) one August date, every filter, linear power, the 3 context boxes
aug = s1_season(TEST_YEAR, "none").filterDate(f"{TEST_YEAR}-08-01", f"{TEST_YEAR}-09-01")
day = ee.Date(ee.Date(aug.first().get("system:time_start")).format("YYYY-MM-dd"))   # start of that day
day_txt = day.format("YYYYMMdd").getInfo()
print("test date:", day_txt)
stack = ee.Image.cat([s1_season(TEST_YEAR, v).filterDate(day, day.advance(1, "day")).mosaic()
                      .select(["VV", "VH"], [f"VV_{v}", f"VH_{v}"]) for v in VARIANTS]).toFloat()
for name, geom in BOX.items():
    export_image(stack, f"S1test_{day_txt}_{name}_lin_10m", geom, crs_transform=GRID10)

# (b) field means for every filter and month of the test year, to compare the link with yield
#     one export per filter, and tileScale = 8 so GEE works in smaller pieces (avoids "memory limit exceeded")
f_test = fc_fields.filter(ee.Filter.eq("year", TEST_YEAR))
for v in VARIANTS:
    coll = s1_season(TEST_YEAR, v)
    tabs = [coll.filterDate(*month_range(TEST_YEAR, m)).mean()
            .reduceRegions(collection=f_test, reducer=ee.Reducer.mean(), scale=10, crs=CRS, tileScale=8)
            .map(tag({"filter": v, "month": m})) for m in MONTHS]
    export_table(ee.FeatureCollection(tabs).flatten(), f"S1test_{TEST_YEAR}_fieldmeans_{v}",
                 ["uid", "year", "filter", "month", "VV", "VH"])
print("filter test exports started:", len(TASKS))

## Step 6: Part A, Sentinel-2

- Collection: `COPERNICUS/S2_SR_HARMONIZED`. Clouds removed with Cloud Score+ (`cs_cdf` of 0.60 or more = clear).
- Indices: NDVI, EVI, GCVI, LSWI, NDRE, NIRv and RECI (the same set as Vaibhav), as the **monthly median of clear observations**, plus `n_clear` (clear observations per pixel).
- A month with no clear observation stays empty (expected in the monsoon).
- The table `S2_clear_per_field_month` gives, for every field and month, the mean and minimum number of clear observations. It shows where optical data is missing.

**Found:** 37 exports (24 monthly field files, 12 NDVI box files and the clear-look table); all completed. The cloud check is in P3b Step 5.

In [ ]:
CS = ee.ImageCollection("GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED")


def s2_season(year):
    def prep(im):
        clear = im.select("cs_cdf").gte(0.60)                      # Cloud Score+: 0.60 or more = clear
        r = im.select(["B2", "B3", "B4", "B5", "B8", "B11"]).divide(10000)
        b2, b3, b4, b5, b8, b11 = [r.select(b) for b in ["B2", "B3", "B4", "B5", "B8", "B11"]]
        ndvi = b8.subtract(b4).divide(b8.add(b4)).rename("NDVI")
        evi = b8.subtract(b4).multiply(2.5).divide(b8.add(b4.multiply(6)).subtract(b2.multiply(7.5)).add(1)).rename("EVI")
        gcvi = b8.divide(b3).subtract(1).rename("GCVI")
        lswi = b8.subtract(b11).divide(b8.add(b11)).rename("LSWI")
        ndre = b8.subtract(b5).divide(b8.add(b5)).rename("NDRE")
        nirv = ndvi.multiply(b8).rename("NIRv")
        reci = b8.divide(b5).subtract(1).rename("RECI")
        return (ee.Image.cat([ndvi, evi, gcvi, lswi, ndre, nirv, reci]).updateMask(clear)
                .set("system:time_start", im.get("system:time_start")))
    return (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED").filterBounds(REGION)
            .filterDate(f"{year}-06-01", f"{year}-12-01").linkCollection(CS, ["cs_cdf"]).map(prep))

def s2_month(coll, year, m):
    c = coll.filterDate(*month_range(year, m))
    return c.median().addBands(c.select("NDVI").count().rename("n_clear")).toFloat()

n0 = len(TASKS)
clear_tabs = []
for yr in YEARS:
    coll = s2_season(yr); mask = field_mask(yr); f_y = fc_fields.filter(ee.Filter.eq("year", yr))
    for m in MONTHS:
        mon = s2_month(coll, yr, m)
        export_image(mon.updateMask(mask), f"S2_{yr}_{m:02d}_fields_10m", REGION, crs_transform=GRID10)
        clear_tabs.append(mon.select("n_clear").unmask(0)
                          .reduceRegions(collection=f_y, reducer=ee.Reducer.mean().combine(ee.Reducer.min(), None, True), scale=10, crs=CRS)
                          .map(tag({"month": m})))
    ndvi_box = ee.Image.cat([s2_month(coll, yr, m).select(["NDVI"], [f"NDVI_{m:02d}"]) for m in MONTHS])
    for name, geom in BOX.items():
        export_image(ndvi_box, f"S2NDVI_{yr}_{name}_10m", geom, crs_transform=GRID10)
export_table(ee.FeatureCollection(clear_tabs).flatten(), "S2_clear_per_field_month", ["uid", "year", "month", "mean", "min"])
print("Sentinel-2 exports started:", len(TASKS) - n0)

## Step 7: Part A, weather, MODIS and static layers

- **MODIS:** LAI and FPAR (MOD15A2H), GPP and net photosynthesis (MOD17A2HGF, or MOD17A2H if the gap-filled version is not out yet), monthly means, 500 m. A 500 m pixel is 25 ha, so this describes the area around a field, not the field.
- **ERA5-Land:** mean and maximum temperature (°C), VPD (kPa, from temperature and dew point), rain, actual and potential evaporation (mm, sign turned positive) and topsoil moisture.
- **CHIRPS:** rain per month and for the season (mm).
- **Terrain:** SRTM elevation, slope, aspect. TWI = ln(a / tan β), with the upstream area from MERIT Hydro (90 m), so it is approximate.
- **SoilGrids:** clay, sand (g/kg), organic carbon (dg/kg) and pH × 10, at 0 to 5, 5 to 15 and 15 to 30 cm.
- **WorldCover 2021:** land cover classes at 10 m.

**Found:** 15 exports (4 MODIS, 4 ERA5-Land, 4 CHIRPS, terrain, soil, land cover). The first MODIS run failed because the GPP band names were wrong (`Gpp_500m` instead of `Gpp`, `PsnNet_500m` instead of `PsnNet`); the names were fixed (as in the cell) and the 4 MODIS files were exported again and completed.

In [ ]:
n0 = len(TASKS)
def season(coll_id, year):
    return ee.ImageCollection(coll_id).filterDate(f"{year}-06-01", f"{year}-12-01")

for yr in YEARS:
    # MODIS LAI and FPAR (8-day, 500 m) and GPP (8-day, 500 m; gap-filled version when available)
    lai = season("MODIS/061/MOD15A2H", yr).map(
        lambda im: im.select(["Lai_500m", "Fpar_500m"]).updateMask(im.select("Lai_500m").lte(100)))
    gf = season("MODIS/061/MOD17A2HGF", yr)
    use_gf = gf.size().getInfo() > 0
    gpp = (gf if use_gf else season("MODIS/061/MOD17A2H", yr)).map(
        lambda im: im.select(["Gpp", "PsnNet"]).updateMask(im.select("Gpp").lte(3000)))    # valid range 0 to 3000; larger = fill codes
    mb = []
    for m in MONTHS:
        l = lai.filterDate(*month_range(yr, m)).mean(); g = gpp.filterDate(*month_range(yr, m)).mean()
        mb += [l.select("Lai_500m").multiply(0.1).rename(f"LAI_{m:02d}"), l.select("Fpar_500m").multiply(0.01).rename(f"FPAR_{m:02d}"),
               g.select("Gpp").multiply(0.0001).rename(f"GPP_{m:02d}"), g.select("PsnNet").multiply(0.0001).rename(f"PSNNET_{m:02d}")]
    export_image(ee.Image.cat(mb).toFloat(), f"MODIS_{yr}_500m", REGION, scale=500)
    print(yr, "MODIS GPP from", "MOD17A2HGF (gap-filled)" if use_gf else "MOD17A2H")

    # ERA5-Land monthly (about 11 km): temperature, VPD, rain, evaporation, soil moisture
    eb = []
    for m in MONTHS:
        im = ee.ImageCollection("ECMWF/ERA5_LAND/MONTHLY_AGGR").filterDate(*month_range(yr, m)).first()
        t, td = im.select("temperature_2m").subtract(273.15), im.select("dewpoint_temperature_2m").subtract(273.15)
        es = lambda x: x.multiply(17.27).divide(x.add(237.3)).exp().multiply(0.6108)    # kPa
        eb += [t.rename(f"T_{m:02d}"), im.select("temperature_2m_max").subtract(273.15).rename(f"TMAX_{m:02d}"),
               es(t).subtract(es(td)).rename(f"VPD_{m:02d}"),
               im.select("total_precipitation_sum").multiply(1000).rename(f"PRECIP_{m:02d}"),
               im.select("total_evaporation_sum").multiply(-1000).rename(f"AET_{m:02d}"),
               im.select("potential_evaporation_sum").multiply(-1000).rename(f"PET_{m:02d}"),
               im.select("volumetric_soil_water_layer_1").rename(f"SM_{m:02d}")]
    export_image(ee.Image.cat(eb).toFloat(), f"ERA5_{yr}_10km", REGION, scale=10000)

    # CHIRPS daily rain (about 5.5 km), summed per month and over the season
    ch = season("UCSB-CHG/CHIRPS/DAILY", yr)
    cb = [ch.filterDate(*month_range(yr, m)).sum().rename(f"RAIN_{m:02d}") for m in MONTHS]
    export_image(ee.Image.cat(cb + [ch.sum().rename("RAIN_season")]).toFloat(), f"CHIRPS_{yr}_5km", REGION, scale=5000)

# static layers: terrain, soil, land cover
dem = ee.Image("USGS/SRTMGL1_003").select("elevation")
slope = ee.Terrain.slope(dem)
upa = ee.Image("MERIT/Hydro/v1_0_1").select("upa")                      # upstream area, km2 (90 m)
twi = upa.multiply(1e6).divide(90).divide(slope.multiply(np.pi / 180).tan().max(0.001)).log()
export_image(ee.Image.cat([dem.rename("ELEV"), slope.rename("SLOPE"), ee.Terrain.aspect(dem).rename("ASPECT"),
                           twi.rename("TWI")]).toFloat(), "DEM_terrain_30m", REGION, scale=30)
soil = ee.Image.cat([ee.Image(f"projects/soilgrids-isric/{p}_mean").select([f"{p}_0-5cm_mean", f"{p}_5-15cm_mean", f"{p}_15-30cm_mean"])
                     for p in ["clay", "sand", "soc", "phh2o"]])
export_image(soil.toFloat(), "SOILGRIDS_250m", REGION, scale=250)
wc = ee.ImageCollection("ESA/WorldCover/v200").first().select("Map").rename("LANDCOVER")
export_image(wc.toByte(), "WORLDCOVER_2021_10m", REGION, crs_transform=GRID10)
print("weather, MODIS and static exports started:", len(TASKS) - n0)

## Step 8: Task status

Run this cell again from time to time. When everything says `COMPLETED`, download the folder `thesis_phase3` from Google Drive into `data/raster/`. You can also follow the tasks at https://code.earthengine.google.com/tasks.

**Found:** all Part A tasks completed (after the MODIS fix). The 61 files were downloaded into `data/raster/` and checked in P3b (0 problems).

In [ ]:
def status():
    st = pd.DataFrame([{"file": n, "state": t.status()["state"], "error": t.status().get("error_message", "")} for n, t in TASKS])
    print(st["state"].value_counts().to_dict())
    bad = st[st["state"].isin(["FAILED", "CANCELLED"])]
    if len(bad): print(bad.to_string(index=False))
    return st
st = status()
st.to_csv(OUT / "P3_export_tasks.csv", index=False)

## Step 9: Part B, main Sentinel-1 exports (after P3b)

`FILTER` is the filter chosen in P3b Step 4. It exports, for each year:
- every month as one file for the fields plus 30 m (6 bands), and
- one file per context box with VV and VH for all six months (12 bands).

**Found:** `FILTER = "leesigma"` (chosen in P3b Step 4). 36 exports started (24 monthly field files and 12 box files) and all completed. P3b Part B checked them: 2 to 3 images per month, about 0% bad values, and the field means match the GEE test table within 0.06 dB.

In [6]:
FILTER = "leesigma"         # chosen in P3b Step 4

if FILTER:
    n0 = len(TASKS)
    for yr in YEARS:
        coll, mask = s1_season(yr, FILTER), field_mask(yr)
        for m in MONTHS:
            export_image(s1_month(coll, yr, m).updateMask(mask), f"S1_{yr}_{m:02d}_{FILTER}_fields_10m", REGION, crs_transform=GRID10)
        box_stack = ee.Image.cat([s1_month(coll, yr, m).select(["VV", "VH"], [f"VV_{m:02d}", f"VH_{m:02d}"]) for m in MONTHS])
        for name, geom in BOX.items():
            export_image(box_stack, f"S1_{yr}_{name}_{FILTER}_10m", geom, crs_transform=GRID10)
    print("main Sentinel-1 exports started:", len(TASKS) - n0)
else:
    print("FILTER not set yet: choose it in P3b first.")

main Sentinel-1 exports started: 36


## Summary

| Part | Exports | Result |
|---|---|---|
| A: filter test | 3 box rasters, 6 tables | completed; P3b chose `leesigma` |
| A: Sentinel-2 | 24 field files, 12 NDVI box files, 1 table | completed |
| A: MODIS, ERA5-Land, CHIRPS, terrain, soil, land cover | 15 files | completed (MODIS after the band-name fix) |
| B: Sentinel-1 main (`leesigma`) | 24 field files, 12 box files | completed and checked in P3b Part B |

All files are in `data/raster/` (not pushed to GitHub). The checks and maps are in P3b.